# 08 · Transition integration, the grouped channels on the learn rate

Runs the grouped transition grid for completeness, joint-trained BKT whose learn rate carries the two grouped states as separate additive suppressors, the emission left plain BKT, so any delta against the engine baseline is the transition's alone:

P(learn at t) = learn_k times (1 minus gamma_conceptual times m_c minus gamma_procedural times m_p), floored at a clip.

Each gamma is a relative suppression of the learn rate per unit of its group's state, descriptive, and the two raw gammas are not on a common scale, the group state distributions differ, means near 0.57 conceptual and 0.28 procedural, so comparisons read at contribution level, gamma times mean state. The chains are GroupedChains under the chain of record, fitted once and shared frozen.

**Expectations, set by notebook 07.** The pooled transition returned gamma effectively zero, free fits at or near the bound with the persistence-absorption mechanism measured, so the free gammas here are expected near zero as well, this grid prices the grouped grain at the transition door and completes the factorial, its informative outcome is either a confirming null or a split the pooled scalar averaged away, a conceptual gamma loading where the pooled one could not.

**The rows.** Validity, both gammas pinned to 0, must reproduce the standing validity trio exactly, 0.6035, 0.6397, 0.5531, the engine's BKT once more. Then free, both gammas fitted independently, and pinned to 1, full additive suppression, where the factor can truncate at the floor when both states are live.

The standing caveats carry, selection on the test set, single-seed deltas wobble by a few thousandths, learn and the gammas compensate through the same expected counts, read them together.

## 1. Setup

In [1]:
import pandas as pd
from scripts.load_data import load_paper_filtered_data
from scripts.chain import TriggerChain
from scripts.misconception_chains_grouped import GroupedChains
from scripts.emission_integration_pooled import CHAIN_OF_RECORD
from scripts.transition_integration_grouped import TransitionIntegrationGrouped

train_df = load_paper_filtered_data("data/mathdial_train.csv")
test_df = load_paper_filtered_data("data/mathdial_test.csv")

chains = GroupedChains(train_df, test_df, chain_class=TriggerChain,
                       chain_kwargs=dict(CHAIN_OF_RECORD))
chains.run()
chains.summary()

,chain,pi,onset,resolve,pP_i,pP_a,expected_dwell_turns,accuracy,tpr,tnr,auc,f1,informative_cells
0,conceptual,0.05,0.0,0.0348,0.02,0.97,28.7,0.6324,0.6683,0.4100,0.5801,0.7579,1439
1,procedural,0.05,0.0,0.0870,0.02,0.97,11.5,0.6739,0.5610,0.8047,0.7250,0.6487,1282


## 2. Rows

Three fits.

In [2]:
ROWS = {
    "validity, gamma=0": {"pin_gamma": 0.0},
    "free":              {},
    "gamma=1":           {"pin_gamma": 1.0},
}

models = {}
for name, kwargs in ROWS.items():
    model = TransitionIntegrationGrouped(train_df, test_df, chains=chains,
                                         **kwargs)
    model.run()
    models[name] = model
    print(f"{name:20s} {model.metrics}")

validity, gamma=0    {'accuracy': 0.6035, 'auc': 0.6397, 'f1': 0.5531, 'turns': 1985, 'gamma_conceptual': 0.0, 'gamma_procedural': 0.0}
free                 {'accuracy': 0.6025, 'auc': 0.6396, 'f1': 0.5489, 'turns': 1985, 'gamma_conceptual': 0.0779, 'gamma_procedural': 0.1865}
gamma=1              {'accuracy': 0.6141, 'auc': 0.643, 'f1': 0.5598, 'turns': 1985, 'gamma_conceptual': 1.0, 'gamma_procedural': 1.0}


## 3. Results table

The frozen M1 trio is the anchor, deltas against it are context, section 4 carries the inference.

In [3]:
M1 = {"accuracy": 0.6065, "auc": 0.6428, "f1": 0.5560}

results = pd.DataFrame(
    [{"row": name, **models[name].metrics} for name in ROWS])
for metric in ("accuracy", "auc", "f1"):
    results[f"d_{metric}"] = (results[metric] - M1[metric]).round(4)
results = results.set_index("row")
results[["gamma_conceptual", "gamma_procedural", "accuracy", "d_accuracy",
         "auc", "d_auc", "f1", "d_f1", "turns"]]

,gamma_conceptual,gamma_procedural,accuracy,d_accuracy,auc,d_auc,f1,d_f1,turns
row,,,,,,,,,
"validity, gamma=0",0.0000,0.0000,0.6035,-0.0030,0.6397,-0.0031,0.5531,-0.0029,1985
free,0.0779,0.1865,0.6025,-0.0040,0.6396,-0.0032,0.5489,-0.0071,1985
gamma=1,1.0000,1.0000,0.6141,0.0076,0.6430,0.0002,0.5598,0.0038,1985


## 4. Deltas against the engine baseline

Both rows against the validity row's three metrics, everything but the transition held fixed. With the free gammas fitted, also read the contributions, gamma times the group's mean state, the scale-comparable quantities.

In [4]:
baseline = models["validity, gamma=0"].metrics
deltas = pd.DataFrame([
    {"row": name,
     "d_accuracy": round(models[name].metrics["accuracy"]
                         - baseline["accuracy"], 4),
     "d_auc": round(models[name].metrics["auc"] - baseline["auc"], 4),
     "d_f1": round(models[name].metrics["f1"] - baseline["f1"], 4)}
    for name in ROWS if name != "validity, gamma=0"])
deltas = deltas.set_index("row").sort_values("d_auc", ascending=False)

import numpy as np
free = models["free"]
states = np.vstack([track[1:, :] for track
                    in chains.predict_states(test_df).values()])
mean_c, mean_p = states.mean(axis=0)
print(f"mean states, conceptual {mean_c:.3f}, procedural {mean_p:.3f}")
print(f"contributions, conceptual "
      f"{free.gamma_conceptual * mean_c:.4f}, procedural "
      f"{free.gamma_procedural * mean_p:.4f}")
deltas

mean states, conceptual 0.547, procedural 0.274
contributions, conceptual 0.0426, procedural 0.0511


,d_accuracy,d_auc,d_f1
row,,,
gamma=1,0.0106,0.0033,0.0067
free,-0.0010,-0.0001,-0.0042


## 5. Interpretation ledger

- The validity row must reproduce the standing validity trio exactly, the fourth cross-script certification of the engine, if it does not, stop and diagnose.
- The gammas are relative suppressions, descriptive, coupled with learn through the same expected counts, a large gamma beside inflated learns is reparameterization. Free gammas here are modest but unstable, multi-start reruns scatter them widely and every free solution carries a worse training likelihood than the nested gamma-zero model, so their combined suppression, around nine per cent of the typical learning rate, does not improve prediction and their individual values are not findings.
- The pooled door's free fit was also initialization-sensitive, an inferior local solution at 0.03 against a bound solution with better likelihood, treat any small nonzero free gamma here with the same suspicion before reading it as signal.
- Under pinned gammas the additive factor can truncate at the floor when both groups are live, the pinned row is a stress case, not an estimate.
- The procedural relationship is empirically opposite to the assumed direction, active procedural state associates with a greater chance of correctness at the next opportunity of the same KC, so the nonnegative-suppression constraint is contradicted for one of the two groups, and the door closes at both grains.

## 6. Why the grouped transition did not improve prediction

The predictive verdict is clean, the free row moves no metric, AUC minus 0.0001, accuracy minus 0.0010, f1 minus 0.0042 against the engine baseline, and even the pinned stress row gains only 0.0002 AUC against the frozen M1. Review analysis then showed the fitted gammas themselves should not be read as findings, for reasons stronger than a simple near-zero.

**The free gammas are unstable and dominated in likelihood.** Multi-start reruns with identical KC initializations land the conceptual gamma anywhere from 0.001 to 0.196 and the procedural from 0.089 to 0.211, and every free solution has a *worse* training likelihood than the nested gamma-zero model, which the free model can approximate but the coordinate procedure fails to reach, different starts settling into different local solutions with nearly identical predictions. The saved 0.078 and 0.187 are path artifacts, their relative sizes uninterpretable, and the best free likelihood sitting at the zero model is itself the strongest statement of the null.

**The combined suppression is modest, not literally zero.** At contribution level the default free model removes roughly nine per cent of the typical learning rate, conceptual and procedural contributions near 0.043 each on training transitions, a real but small factor that the compensating learn rates, mean 0.349 to 0.376, largely absorb, and that buys no ranking.

**The procedural direction is empirically reversed.** Checking whether the current grouped state predicts correctness at the next occurrence of the same KC, conceptual activity shows a weak suppression-shaped association, next-correct 38 against 44 per cent on train, fading to AUC 0.535 on test, while procedural activity points the *opposite* way, active procedural state associates with a greater chance of next-opportunity correctness, test AUC 0.452 for the suppression direction. The model imposes nonnegative suppression on both groups, and the data contradicts that constraint for one of them, the positive fitted procedural gamma reading as latent-mastery confounding, cross-group correlation, and optimizer path rather than any procedural learning impediment.

**The structural weaknesses carry from the earlier verdicts.** The states are nearly dialogue-level constants, 84 to 90 per cent between-dialogue variance, absorbable into per-KC learn rates. The grouped latents were weakened before arriving, the merge's lone-A compromise is nearly the only A case, over ninety-nine per cent single-member on conceptual, and the grouped pre-turn states barely predict correctness at all, 0.56 and 0.52. And the connection is blunt, any group activity suppresses every KC on the turn, where a real transition effect plausibly needs specific misconceptions linked to related KCs.

**The pinned row is arithmetic, not evidence.** With both gammas at one the median suppression factor is 0.132, nearly nine per cent of transitions clip at the floor, mean learn compensates to 0.504, the training likelihood collapses by forty nats, and the 0.0033 AUC nudge spans zero under a dialogue bootstrap, a stress case behaving as labeled.

**Conclusion.** Grouped transition integration closes with the transition door as a whole, the free model improves nothing, the zero model carries the best likelihood, the gamma estimates are initialization artifacts, and one of the two assumed suppression directions is empirically reversed. The stacked grouped-transition model is not built.

## 7. Notes

- Every model keeps its fitted parameters at `.parameters` and the shared frozen chains at `.chains`, nothing here refits the chains.
- No trajectory diagnostic here, notebook 07's was shown to be dominated by composition, a dozen unusually correct early-death dialogues, and nothing in this grid changes that population.
- The outcome, either way, goes to the report's transition verdict alongside notebook 07's.